# RawDigit viewer

Interactive browser for the `RawDigitAna` TTree (one entry per channel):

| branch | type |
|---|---|
| `run`, `subrun`, `event` | int |
| `channel` | int |
| `view` | int (0 = U, 1 = V, 2 = Z) |
| `tpc` | `vector<int>` (all TPCs the channel touches) |
| `adc` | `vector<short>` |


In [ ]:
from collections import defaultdict

import ROOT

ROOT.gStyle.SetOptStat(0)
%jsroot on

## Configuration

Edit these and re-run the cells below.

In [ ]:
FILE_NAME = "prod_muminus_0.1-2.0GeV_isotropic_dune10kt_1x2x6_gen_g4_detsim_rawdigit.root"                 # your RawDigitAna output file

TREE_PATH = "rawdigit/rawdigitTree"       # TFileService directory / tree name

RUN   = 20000015
EVENT = 1

VIEW_NAMES = {0: "U (induction 1)", 1: "V (induction 2)", 2: "Z (collection)"}

In [ ]:
tfile = ROOT.TFile.Open(FILE_NAME)
assert tfile and not tfile.IsZombie(), f"Cannot open {FILE_NAME}"

tree = tfile.Get(TREE_PATH)
if not tree:
    print(f"Tree '{TREE_PATH}' not found -- file contents:")
    tfile.ls()
    raise KeyError(TREE_PATH)

print(f"Opened {FILE_NAME}: {tree.GetEntries()} entries")

## Build an event index (one pass)

Maps `(run, event) -> list of entry numbers` and `(run, event, channel) -> entry number`, so every later lookup is O(1) instead of a full tree scan.

In [ ]:
event_index   = defaultdict(list)   # (run, event)          -> [entry numbers]
channel_index = {}                  # (run, event, channel) -> entry number

for i in range(tree.GetEntries()):
    tree.GetEntry(i)
    key = (int(tree.run), int(tree.event))
    event_index[key].append(i)
    channel_index[(key[0], key[1], int(tree.channel))] = i

print(f"Indexed {len(event_index)} (run, event) pairs")
for (r, e), entries in sorted(event_index.items()):
    print(f"  run {r}  event {e}: {len(entries)} channels")

In [ ]:
def channels_in_tpc(run, event, tpc):
    """Return {view: sorted list of channels} for one TPC in one event."""
    out = defaultdict(list)
    for i in event_index.get((run, event), []):
        tree.GetEntry(i)
        if tpc in list(tree.tpc):
            out[int(tree.view)].append(int(tree.channel))
    return {v: sorted(chs) for v, chs in out.items()}


## Three-view waveform display

Two-step workflow: first inspect which channels are available in a TPC, then choose one channel per view to plot.

**Step 1 — `list_tpc_channels(run, event, tpc)`**
Prints, for the given TPC, the number of channels in each view (U, V, Z) and their global channel ranges. Returns `{view: [channels]}`.

**Step 2 — `plot_chosen_channels(run, event, tpc, ch_u, ch_v, ch_z)`**
Draws the chosen channel of each view in a 1x3 canvas (one pad per view). Channels are **global channel numbers**, taken from the ranges printed in step 1.

Notes:
* Each channel is validated against the requested view and TPC; a mismatch prints a warning and skips that pad instead of plotting the wrong waveform.
* Wrapped induction channels belong to two TPCs -- the pad title shows the full TPC list.
* Waveforms are drawn as raw ADC vs tick (no pedestal subtraction), so the y-axis sits at the pedestal level and signals appear as excursions on top of it.

In [ ]:
_keepalive = []
def list_tpc_channels(run, event, tpc):
    """
    Step 1: show how many channels each view has in this TPC,
    with their global channel ranges. Returns {view: [channels]}.
    """
    per_view = channels_in_tpc(run, event, tpc)
    if not per_view:
        print(f"No channels found in TPC {tpc} for run {run}, event {event}")
        return {}

    print(f"Run {run}  Event {event}  TPC {tpc}")
    print(f"{'view':>5}  {'plane':<18} {'#channels':>10}  {'range':>15}")
    print("-" * 55)
    for view in sorted(per_view):
        chs = per_view[view]
        name = VIEW_NAMES.get(view, f"view {view}")
        print(f"{view:>5}  {name:<18} {len(chs):>10}  "
              f"{chs[0]:>6} - {chs[-1]}")
    print("-" * 55)
    return per_view


def plot_chosen_channels(run, event, tpc, ch_u, ch_v, ch_z):
    """
    Step 2: plot the user-chosen channel for each view (U, V, Z),
    validating that each belongs to the requested TPC and view.
    """
      # keep canvases/histograms alive across cells

    requested = {0: ch_u, 1: ch_v, 2: ch_z}

    canvas = ROOT.TCanvas(f"c_pick_tpc{tpc}_r{run}_e{event}_{len(_keepalive)}",
                          f"Run {run} Event {event} TPC {tpc}", 1000, 900)
    canvas.Divide(1, 3)

    hists = []
    for ipad, view in enumerate(sorted(requested), start=1):
        ch = requested[view]
        canvas.cd(ipad)

        i = channel_index.get((run, event, ch))
        if i is None:
            print(f"  view {view}: channel {ch} not found in this event")
            continue
        tree.GetEntry(i)

        # validation against the user's stated intent
        if int(tree.view) != view:
            print(f"  WARNING: channel {ch} is view {int(tree.view)}, "
                  f"not view {view} -- skipping")
            continue
        tpcs = list(tree.tpc)
        if tpc not in tpcs:
            print(f"  WARNING: channel {ch} belongs to TPCs {tpcs}, "
                  f"not TPC {tpc} -- skipping")
            continue

        adc = list(tree.adc)
        nticks = len(adc)
        name = VIEW_NAMES.get(view, f"view {view}")
        h = ROOT.TH1F(f"h_pick_tpc{tpc}_v{view}_ch{ch}_{len(_keepalive)}",
                      f"{name}  ch {ch}  TPC {tpcs};tick;ADC",
                      nticks, 0, nticks)
        h.SetDirectory(0)
        for j, a in enumerate(adc):
            h.SetBinContent(j + 1, a)
        h.SetLineColor(ROOT.kBlue + 1)
        h.Draw("hist")
        hists.append(h)

    _keepalive.append((canvas, hists))
    canvas.Draw()
    return canvas

In [ ]:
per_view = list_tpc_channels(RUN, EVENT, tpc=0)

In [ ]:
plot_chosen_channels(RUN, EVENT, tpc=0, ch_u=120, ch_v=900, ch_z=1800)

In [ ]:
_keepalive[-1][0].SaveAs(f'waveforms_r{RUN}_e{EVENT}.png')